In [40]:
import pandas as pd
import sys
import os

# Add project root to path
sys.path.append(os.path.abspath(".."))

# Import reusable hypothesis test functions
from src.hypothesis_tests import chi_square_test, t_test


# =========================================================
# 1. LOAD DATA
# =========================================================

df = pd.read_csv("../data/insurance_data_cleaned.csv")

In [41]:
# =========================================================
# 2. FEATURE ENGINEERING
# =========================================================

# Claim Frequency Indicator
df["HasClaim"] = (df["TotalClaims"] > 0).astype(int)

# Margin
df["Margin"] = df["TotalPremium"] - df["TotalClaims"]



In [42]:
# =========================================================
# 3. CONTROL FOR CONFOUNDING VARIABLES
# =========================================================
# We compare groups with similar:
# - VehicleType
# - CoverType
#
# This helps isolate the effect of the feature being tested.

addis_df = df[df["Province"] == "Addis Ababa"]

most_common_vehicle = addis_df["VehicleType"].mode()[0]
most_common_cover = addis_df["CoverType"].mode()[0]

df_controlled = df[
    (df["VehicleType"] == most_common_vehicle) &
    (df["CoverType"] == most_common_cover)
].copy()

# Claim severity dataset
severity_controlled_df = df_controlled[
    df_controlled["TotalClaims"] > 0
].copy()



In [43]:
# =========================================================
# 4. SAMPLE SIZE VALIDATION
# =========================================================

print("\n================ SAMPLE SIZE CHECK ================\n")

print(f"Original Dataset Size: {len(df)}")
print(f"Controlled Dataset Size: {len(df_controlled)}")

print("\nProvince Counts:")
print(df_controlled["Province"].value_counts())

print("\nZip Code Counts:")
print(df_controlled["ZipCode"].value_counts())

print("\nGender Counts:")
print(df["Gender"].value_counts())



================ SAMPLE SIZE CHECK ================

Original Dataset Size: 10000
Controlled Dataset Size: 1607

Province Counts:
Province
Addis Ababa    577
Oromia         385
Amhara         327
Somali         196
Tigray         122
Name: count, dtype: int64

Zip Code Counts:
ZipCode
10004    126
10002    125
10003    111
10001    108
10005    107
20002     83
20004     83
20005     76
20001     74
30005     72
30003     70
20003     69
30002     64
30001     61
30004     60
40004     47
40003     42
40005     37
40001     35
40002     35
50003     33
50005     26
50004     24
50002     22
50001     17
Name: count, dtype: int64

Gender Counts:
Gender
Female    5138
Male      4862
Name: count, dtype: int64


In [44]:

# =========================================================
# 5. HYPOTHESIS TESTS
# =========================================================

print("\n================ RUNNING HYPOTHESIS TESTS ================\n")

# ---------------------------------------------------------
# H0: No risk difference across provinces
# KPI: Claim Frequency
# Test: Chi-square
# ---------------------------------------------------------

province_freq = chi_square_test(
    df=df_controlled,
    group_col="Province",
    target_col="HasClaim",
    group_a="Addis Ababa",
    group_b="Oromia"
)

# Claim frequency by province
province_claim_rates = (
    df_controlled[df_controlled["Province"].isin(["Addis Ababa", "Oromia"])]
    .groupby("Province")["HasClaim"]
    .mean()
)

print("\nProvince Claim Frequencies:")
print(province_claim_rates)


# ---------------------------------------------------------
# H0: No risk difference between zip codes
# KPI: Claim Severity
# Test: Welch T-Test
# ---------------------------------------------------------

# Using claim severity as the risk KPI

zip_severity = t_test(
    df=severity_controlled_df,
    group_col="ZipCode",
    metric_col="TotalClaims",
    group_a=10004,
    group_b=10002,
)

zip_claim_means = (
    severity_controlled_df[
        severity_controlled_df["ZipCode"].isin([10004, 10002])
    ]
    .groupby("ZipCode")["TotalClaims"]
    .mean()
)

print("\nZip Code Claim Severity Means:")
print(zip_claim_means)


# ---------------------------------------------------------
# H0: No margin difference between zip codes
# KPI: Margin
# Test: Welch T-Test
# ---------------------------------------------------------

zip_margin = t_test(
    df=df_controlled,
    group_col="ZipCode",
    metric_col="Margin",
    group_a=10004,
    group_b=10002
)

zip_margin_means = (
    df_controlled[
        df_controlled["ZipCode"].isin([10004, 10002])
    ]
    .groupby("ZipCode")["Margin"]
    .mean()
)

print("\nZip Code Margin Means:")
print(zip_margin_means)


# ---------------------------------------------------------
# H0: No risk difference between Men and Women
# KPI: Claim Frequency
# Test: Chi-square
# ---------------------------------------------------------

gender_test = chi_square_test(
    df=df,
    group_col="Gender",
    target_col="HasClaim",
    group_a="Male",
    group_b="Female"
)

gender_claim_rates = (
    df.groupby("Gender")["HasClaim"]
    .mean()
)

print("\nGender Claim Frequencies:")
print(gender_claim_rates)



================ RUNNING HYPOTHESIS TESTS ================


Province Claim Frequencies:
Province
Addis Ababa    0.116118
Oromia         0.116883
Name: HasClaim, dtype: float64

Zip Code Claim Severity Means:
ZipCode
10002    5843.277778
10004    7170.250000
Name: TotalClaims, dtype: float64

Zip Code Margin Means:
ZipCode
10002    1399.448000
10004    1318.031746
Name: Margin, dtype: float64

Gender Claim Frequencies:
Gender
Female    0.153756
Male      0.153229
Name: HasClaim, dtype: float64


In [45]:

# =========================================================
# 6. RESULTS SUMMARY TABLE
# =========================================================

results = pd.DataFrame([
    {
        "Hypothesis": "Province Risk Difference",
        "KPI": "Claim Frequency",
        "Test": province_freq["test"],
        "P-Value": round(province_freq["p_value"], 6),
        "Decision": (
            "Reject H0"
            if province_freq["reject_null"]
            else "Fail to Reject H0"
        )
    },
    {
        "Hypothesis": "Zip Code Risk Difference",
        "KPI": "Claim Severity",
        "Test": zip_severity["test"],
        "P-Value": round(zip_severity["p_value"], 6),
        "Decision": (
            "Reject H0"
            if zip_severity["reject_null"]
            else "Fail to Reject H0"
        )
    },
    {
        "Hypothesis": "Zip Code Margin Difference",
        "KPI": "Margin",
        "Test": zip_margin["test"],
        "P-Value": round(zip_margin["p_value"], 6),
        "Decision": (
            "Reject H0"
            if zip_margin["reject_null"]
            else "Fail to Reject H0"
        )
    },
    {
        "Hypothesis": "Gender Risk Difference",
        "KPI": "Claim Frequency",
        "Test": gender_test["test"],
        "P-Value": round(gender_test["p_value"], 6),
        "Decision": (
            "Reject H0"
            if gender_test["reject_null"]
            else "Fail to Reject H0"
        )
    }
])


print("\n================ FINAL RESULTS ================\n")
print(results.to_string(index=False))



================ FINAL RESULTS ================

                Hypothesis             KPI         Test  P-Value          Decision
  Province Risk Difference Claim Frequency   Chi-Square 1.000000 Fail to Reject H0
  Zip Code Risk Difference  Claim Severity Welch T-Test 0.305539 Fail to Reject H0
Zip Code Margin Difference          Margin Welch T-Test 0.796826 Fail to Reject H0
    Gender Risk Difference Claim Frequency   Chi-Square 0.963831 Fail to Reject H0


In [46]:
# =========================================================
# 7. BUSINESS INTERPRETATIONS
# =========================================================

print("\n================ BUSINESS INTERPRETATIONS ================\n")

for _, row in results.iterrows():

    if row["Decision"] == "Reject H0":

        print(f"""
{row['Hypothesis']}:
We reject the null hypothesis because p-value ({row['P-Value']})
is less than 0.05.

This suggests a statistically significant difference exists.

Business Recommendation:
ACIS should consider segment-specific pricing,
underwriting adjustments, or targeted risk controls.
""")

    else:

        print(f"""
{row['Hypothesis']}:
We fail to reject the null hypothesis because p-value
({row['P-Value']}) is greater than 0.05.

This suggests no statistically significant difference
was detected between the compared groups.

Business Recommendation:
Current pricing and segmentation strategies may already
be adequate for these groups. Additional data or broader
feature engineering may improve future analyses.
""")


================ BUSINESS INTERPRETATIONS ================


Province Risk Difference:
We fail to reject the null hypothesis because p-value
(1.0) is greater than 0.05.

This suggests no statistically significant difference
was detected between the compared groups.

Business Recommendation:
Current pricing and segmentation strategies may already
be adequate for these groups. Additional data or broader
feature engineering may improve future analyses.


Zip Code Risk Difference:
We fail to reject the null hypothesis because p-value
(0.305539) is greater than 0.05.

This suggests no statistically significant difference
was detected between the compared groups.

Business Recommendation:
Current pricing and segmentation strategies may already
be adequate for these groups. Additional data or broader
feature engineering may improve future analyses.


Zip Code Margin Difference:
We fail to reject the null hypothesis because p-value
(0.796826) is greater than 0.05.

This suggests no statistica